# C3 - Day 9: Model 3 - read an unfinished document (QLoRA)
**AnchorAI | Component 3: Cognitive Checkpointing**

**Task.** A student was writing a text document (report, essay, notes, proposal) and was interrupted. The model reads the
unfinished text and answers with a short JSON record:
* **stopped_at**: the section the student was in and the last words written (copied from the document),
* **remaining**: the sections still to do, and where that information comes from (the student's own plan, the student's own
  headings, or the usual structure of that kind of document),
* **search_queries**: one web-search phrase per open section. The app turns them into search links; **nothing is generated for the document**.

The model never writes any part of the document. That keeps C3 inside its boundary with C4.

**Data.** `data/generated/doc_progress_{train,val,test}.jsonl` (640 / 64 / 128 synthetic documents; topics differ between splits).
Four kinds of document: with a plan line, with TODO headings, plain, and **without any headings** (the model must recognise
what the last paragraph is about). Rules cannot solve the last two kinds, which is why a learned model is needed.

> **Runtime -> Change runtime type -> T4 GPU.** Upload the three `doc_progress_*.jsonl` files to `MyDrive/C3/data/generated/` first.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU found. Use Runtime -> Change runtime type -> T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
!pip -q install peft bitsandbytes accelerate

In [ ]:
import os, json, random, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/C3'
GEN_DIR = os.path.join(BASE, 'data', 'generated')
OUTPUT_DIR = os.path.join(BASE, 'outputs')
ADAPTER_DIR = os.path.join(BASE, 'qwen_c3_docs_lora')
os.makedirs(OUTPUT_DIR, exist_ok=True)
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

def read_jsonl(name):
    with open(os.path.join(GEN_DIR, name), encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = read_jsonl('doc_progress_train.jsonl')
val_rows = read_jsonl('doc_progress_val.jsonl')
print(len(train_rows), 'training documents,', len(val_rows), 'validation documents')
print(train_rows[0]['input']); print('--- expected answer ---'); print(train_rows[0]['output'])

## Helper code (the same files the local tests run)

In [ ]:
"""LoRA training and generation helpers for Model 2.

This file is used twice: the Colab notebooks include its source, and the local tests run the same code
on a tiny random model, so a mistake here is caught before any GPU time is spent.
"""
import inspect
import time

import torch
from torch.utils.data import Dataset
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments, set_seed

LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


def encode_row(tok, row, max_len=1024):
    """Tokenise one chat example; only the assistant answer is trained on (the prompt is masked with -100)."""
    prompt = tok.apply_chat_template(row["messages"][:2], tokenize=False, add_generation_prompt=True)
    full = tok.apply_chat_template(row["messages"], tokenize=False)
    if not full.startswith(prompt):
        raise ValueError("chat template does not start with the prompt for " + row.get("id", "?"))
    prompt_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    full_ids = tok(full, add_special_tokens=False)["input_ids"]
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    n = min(len(full_ids), max_len)
    return {"input_ids": full_ids[:n], "attention_mask": [1] * n, "labels": labels[:n]}


class ListDataset(Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


def attach_lora(model, quantized, r=16, alpha=32, dropout=0.05):
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    if quantized:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    config = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout, bias="none", task_type="CAUSAL_LM",
                        target_modules=LORA_TARGETS)
    return get_peft_model(model, config)


def training_arguments(output_dir, epochs, lr, batch_size, accum, has_eval, fp16, seed):
    kwargs = dict(
        output_dir=output_dir, num_train_epochs=epochs, per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size, gradient_accumulation_steps=accum, learning_rate=lr,
        lr_scheduler_type="cosine", warmup_steps=5, logging_steps=5, save_strategy="no", report_to="none",
        fp16=fp16, seed=seed, remove_unused_columns=False,
    )
    if has_eval:
        accepted = inspect.signature(TrainingArguments.__init__).parameters
        kwargs["eval_strategy" if "eval_strategy" in accepted else "evaluation_strategy"] = "epoch"
    return TrainingArguments(**kwargs)


def train(model, tok, train_rows, val_rows, out_dir, epochs=5, lr=2e-4, batch_size=4, accum=4, fp16=True,
          max_len=1024, seed=42):
    set_seed(seed)
    train_items = [encode_row(tok, r, max_len) for r in train_rows]
    val_items = [encode_row(tok, r, max_len) for r in val_rows]
    collator = DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100)
    args = training_arguments(out_dir, epochs, lr, batch_size, accum, bool(val_items), fp16, seed)
    trainer = Trainer(model=model, args=args, train_dataset=ListDataset(train_items),
                      eval_dataset=ListDataset(val_items) if val_items else None, data_collator=collator)
    model.config.use_cache = False
    start = time.time()
    result = trainer.train()
    model.config.use_cache = True
    info = {
        "seconds": round(time.time() - start, 1), "train_loss": float(result.training_loss),
        "epochs": epochs, "learning_rate": lr, "batch_size": batch_size, "grad_accumulation": accum,
        "max_len": max_len, "seed": seed, "n_train": len(train_items), "n_val": len(val_items),
        "longest_example_tokens": max(len(i["input_ids"]) for i in train_items),
        "history": trainer.state.log_history,
    }
    return trainer, info


@torch.no_grad()
def generate_batch(model, tok, message_lists, max_new_tokens=300, batch_size=8):
    """Greedy generation for a list of chat prompts (system + user messages); returns the answers."""
    model.eval()
    tok.padding_side = "left"
    answers = []
    for i in range(0, len(message_lists), batch_size):
        chunk = message_lists[i:i + batch_size]
        prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in chunk]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
        for j in range(len(chunk)):
            new_tokens = out[j][enc["input_ids"].shape[1]:]
            answers.append(tok.decode(new_tokens, skip_special_tokens=True).strip())
    return answers


## Load the model in 4-bit and attach LoRA
Same method as Model 2: 4-bit NF4 base model, small trainable LoRA matrices.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb, device_map='auto')
model = attach_lora(model, quantized=True)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {trainable:,} ({100 * trainable / sum(p.numel() for p in model.parameters()):.2f}%)')

## Train
4 passes over the data. The prompt is masked, so the model is trained only on the JSON answer. Validation loss is computed after each epoch (about 20 to 30 minutes on a T4).

In [ ]:
trainer, info = train(model, tok, train_rows, val_rows, os.path.join(BASE, 'qwen_c3_docs_runs'),
                      epochs=4, lr=2e-4, batch_size=4, accum=4, fp16=True, max_len=1280, seed=42)
print(f"Done in {info['seconds']} s | mean training loss {info['train_loss']:.3f} | longest example {info['longest_example_tokens']} tokens")

In [ ]:
hist = info['history']
tl = [(h['step'], h['loss']) for h in hist if 'loss' in h]
vl = [(h['step'], h['eval_loss']) for h in hist if 'eval_loss' in h]
plt.figure(figsize=(7, 3.8))
plt.plot(*zip(*tl), label='training loss')
if vl: plt.plot(*zip(*vl), 'o-', label='validation loss')
plt.xlabel('step'); plt.ylabel('loss'); plt.title('Model 3 - QLoRA training'); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model3_training_loss.png'), dpi=150); plt.show()
print('validation loss per epoch:', [round(v, 3) for _, v in vl])

## Save the adapter to Google Drive (not to git)

In [ ]:
model.save_pretrained(ADAPTER_DIR); tok.save_pretrained(ADAPTER_DIR)
meta = {k: v for k, v in info.items() if k != 'history'}
meta.update({'base_model': MODEL_NAME, 'method': 'QLoRA (4-bit NF4, LoRA r=16, alpha=32)', 'data': 'synthetic doc_progress_train.jsonl'})
json.dump(meta, open(os.path.join(ADAPTER_DIR, 'training_info.json'), 'w'), indent=2)
print('Saved to', ADAPTER_DIR, sorted(os.listdir(ADAPTER_DIR)))

## Quick look at two validation documents

In [ ]:
sample = [r for r in val_rows if r['variant'] == 'headingless'][:1] + [r for r in val_rows if r['variant'] == 'plain'][:1]
answers = generate_batch(model, tok, [r['messages'][:2] for r in sample], max_new_tokens=500, batch_size=2)
for r, a in zip(sample, answers):
    print('=' * 80); print(r['variant']); print(r['document']); print('--- model ---'); print(a)